# 🧬 Pipeline A: Raw Sequence → Improved CNN (Baseline)

## Architecture
```
DNA Sequence (101bp string)
       │
       ▼
┌──────────────────────────┐
│   Base Integer Mapping   │  A=0, C=1, G=2, T=3, N=4
│   → (batch, 101)         │
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│   Learnable Embedding    │  Maps integers to continuous 32-dim vectors
│   → (batch, 101, 32)     │
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│   Multi-Scale Conv1D     │  kernel_sizes=[3,5,7,9], channels=64
│   with Branch Dropout    │  (Learns motif filters)
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│ Global Max + Avg Pooling │  MAX: motif presence
│  → (batch, 512)          │  AVG: motif abundance (GC content / frequency)
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│    Classification Head   │  FC (512 -> 64) -> Dropout(0.6) -> FC (64 -> 4)
└──────────────────────────┘
       │
       ▼
  SP1 / SP2 / SP4 / Negative
```

### How does this solve overfitting?
1. **Learnable Embeddings**: Allows the model to learn relationships between nucleotide characters instead of using sparse, disconnected one-hot arrays.
2. **Global MAX + AVG Pooling**: Global Average Pooling acts as a learnable k-mer frequency Counter (matching the strength of the CountVectorizer baseline), while Global Max Pooling acts as a motif detector.
3. **Regularization**: Intermediate branch dropout (0.2) + strong final classification dropout (0.6) + weight decay (1e-4).
4. **Low Parameter Count**: Total parameters are only **~85K**, preventing the model from memorizing the training set.

⚡ **Fast to train**: Runs in ~30 seconds on CPU, even faster on GPU.

### Cell 1: Setup

In [ ]:
!pip install -q scikit-learn matplotlib seaborn

import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

if os.path.basename(os.getcwd()) == REPO_NAME:
    os.chdir("..")

if not os.path.isdir(REPO_NAME):
    print("Cloning...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"Ready: {os.getcwd()}")

### Cell 2: Load Data + Map to Integers

In [ ]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from src.mcnn_model import ImprovedOneHotCNN
from src.train import train_model, evaluate_model, plot_curves

# --- Load FASTA ---
def load_fasta(path):
    seqs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

# --- Map nucleotides to integer indices (A=0, C=1, G=2, T=3, N/Others=4) ---
def seqs_to_indices(sequences):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    n = len(sequences)
    seq_len = len(sequences[0])
    indices = np.zeros((n, seq_len), dtype=np.int64)
    for i, seq in enumerate(sequences):
        for j, nuc in enumerate(seq):
            indices[i, j] = mapping.get(nuc, 4)
    return indices

print("Loading datasets...")
seqs_sp1 = load_fasta("data/processed/sp1_positive_final.fasta")
seqs_sp2 = load_fasta("data/processed/sp2_positive_final.fasta")
seqs_sp4 = load_fasta("data/processed/sp4_positive_final.fasta")
seqs_neg = load_fasta("data/processed/negative_final.fasta")

sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
y = np.concatenate([
    np.zeros(len(seqs_sp1)),
    np.ones(len(seqs_sp2)),
    np.full(len(seqs_sp4), 2),
    np.full(len(seqs_neg), 3)
])

print(f"Total: {len(sequences)} sequences")
print(f"Distribution: {dict(zip(['SP1','SP2','SP4','Neg'], np.bincount(y.astype(int))))}")

# --- Integer encode ---
print("\nMapping sequences to integer indices...")
X = seqs_to_indices(sequences)
print(f"Feature tensor shape: {X.shape}  (samples, position_indices)")

### Cell 3: Build Improved Model + DataLoaders

In [ ]:
# --- Stratified Split ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}")

# --- TensorDatasets ---
train_ds = TensorDataset(
    torch.from_numpy(X_train).long(),
    torch.from_numpy(y_train).long()
)
val_ds = TensorDataset(
    torch.from_numpy(X_val).long(),
    torch.from_numpy(y_val).long()
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")

# --- Initialize ImprovedOneHotCNN ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = ImprovedOneHotCNN(
    seq_len=101,
    num_classes=4,
    embedding_dim=32,       # continuous space representation
    branch_channels=64,     # lightweight filter footprint
    kernel_sizes=[3, 5, 7, 9],
    dropout_rate=0.6        # strict regularization
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,} (Very lightweight)")

### Cell 4: Train

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=25,
    lr=0.002,               # Higher learning rate since we have embedding layers
    device=device,
    output_dir='models'
)

### Cell 5: Evaluate + Plots

In [ ]:
from IPython.display import Image, display
import os

# Training curves
plot_curves(history, save_dir='figures')
display(Image('figures/mcnn_training_curves.png'))

# Load best checkpoint and evaluate
best_path = os.path.join('models', 'best_mcnn_model.pt')
model.load_state_dict(torch.load(best_path, map_location=device))
print(f"Loaded best model from {best_path}")

class_names = ['SP1', 'SP2', 'SP4', 'Negative']
evaluate_model(model, val_loader, class_names, device=device, save_dir='figures')

print("\n--- Confusion Matrix ---")
display(Image('figures/confusion_matrix.png'))
print("\n--- ROC Curves ---")
display(Image('figures/roc_curves.png'))
print("\n--- Precision-Recall Curves ---")
display(Image('figures/precision_recall_curves.png'))